# NutriScan — YOLOv8 Training on UEC FOOD-256

**Authors** Yu Cao, Na Yin, Fan Zhang
**Date:**April 2026

**Runtime:** GPU (T4 or better recommended)  
**Steps:**
1. Mount Google Drive
2. Install ultralytics
3. Extract `dataset256.zip` from `MyDrive/nutriscan/`
4. Convert UEC FOOD-256 `bb_info.txt` bounding boxes to YOLO format
5. Build `data.yaml`
6. Train YOLOv8n for 50 epochs
7. Save `best.pt` to Drive

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nutriscan'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

Mounted at /content/drive
Drive directory: /content/drive/MyDrive/nutriscan


## 2 — Install ultralytics

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 43.1/235.7 GB disk)


## 3 — Extract UEC FOOD-256

Place `dataset256.zip` (downloaded from http://foodcam.mobi/dataset256.html) in your
Google Drive at `MyDrive/nutriscan/dataset256.zip`, then run this cell.

The zip extracts to a folder named `UECFOOD256/` containing 256 numbered sub-folders
(one per food category). Each sub-folder contains `.jpg` images and a `bb_info.txt`
bounding-box file.

In [ ]:
import zipfile
import pathlib

DATASET_ROOT = pathlib.Path('/content/UECFOOD256')
ARCHIVE_PATH = pathlib.Path(DRIVE_DIR) / 'dataset256.zip'

if not DATASET_ROOT.exists():
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f'Archive not found: {ARCHIVE_PATH}\n'
            'Download dataset256.zip from http://foodcam.mobi/dataset256.html '
            f'and upload it to {DRIVE_DIR}/'
        )
    print(f'Extracting {ARCHIVE_PATH} ...')
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall('/content/')
    print('Extraction complete.')

    # The zip may place files directly under /content/UECFOOD256/ or under a
    # nested folder.  Find the actual root by locating any bb_info.txt.
    if not DATASET_ROOT.exists():
        candidates = list(pathlib.Path('/content').rglob('bb_info.txt'))
        if not candidates:
            raise RuntimeError('bb_info.txt not found after extraction — check the zip structure.')
        # Walk up to the first directory that contains numbered sub-folders
        for candidate in candidates:
            parent = candidate.parent.parent  # e.g. /content/SomeName/1/ -> /content/SomeName
            if parent.name != 'UECFOOD256':
                print(f'Renaming {parent} → {DATASET_ROOT}')
                parent.rename(DATASET_ROOT)
            break

# Verify
cat_dirs = sorted(
    [d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda p: int(p.name),
)
print(f'Found {len(cat_dirs)} category folders (expected 256).')
if len(cat_dirs) < 256:
    print('WARNING: fewer than 256 folders — zip may be incomplete.')

Extracting /content/drive/MyDrive/nutriscan/dataset256.zip ...
Extraction complete.
Found 256 category folders (expected 256).


## 4 — Convert `bb_info.txt` Annotations → YOLO Format

UEC FOOD-256 stores bounding boxes in a `bb_info.txt` file inside each numbered
category folder. The format is **space-separated with a single header row**:

```
img    x1    y1    x2    y2
1      52    36    253   317
3      16     1    500   374
```

- `img` — image file stem (e.g. `1` → `1.jpg`)
- `x1 y1` — top-left corner in pixels (1-indexed)
- `x2 y2` — bottom-right corner in pixels (inclusive)
- One image can have **multiple rows** (multiple instances of the same food)

Category names come from `UECFOOD256/category.txt` at the dataset root:
each line is `<folder_number> <food name>`.  Falls back to the folder number
as the class name if the file is absent.

In [ ]:
import random
import shutil
from pathlib import Path
from PIL import Image

YOLO_ROOT = Path('/content/uec_yolo')
for split in ('train', 'val'):
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

# ── Build class name map ────────────────────────────────────────────────────
# category.txt lives at the dataset root; format: "<folder_num> <food name>"
# e.g.  "1 rice"  "2 eels on rice"
class_names: list[str] = []
cat_id_map: dict[str, int] = {}  # folder name (str) -> YOLO class index

root_cat_file = DATASET_ROOT / 'category.txt'
name_by_folder: dict[str, str] = {}

if root_cat_file.exists():
    for line in root_cat_file.read_text(encoding='utf-8', errors='replace').splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)          # split on first whitespace only
        if len(parts) == 2:
            folder_num, food_name = parts[0], parts[1].strip()
            name_by_folder[folder_num] = food_name.replace(' ', '_')

# Sort folders numerically (avoids lexicographic order: 10 < 2)
sorted_cat_dirs = sorted(
    [d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda p: int(p.name),
)

for i, cat_dir in enumerate(sorted_cat_dirs):
    # Prefer root-level category.txt; fall back to per-folder category.txt;
    # last resort: use the folder number itself.
    if cat_dir.name in name_by_folder:
        name = name_by_folder[cat_dir.name]
    else:
        per_folder_cat = cat_dir / 'category.txt'
        if per_folder_cat.exists():
            name = per_folder_cat.read_text(encoding='utf-8', errors='replace').strip().replace(' ', '_')
        else:
            name = cat_dir.name
    class_names.append(name)
    cat_id_map[cat_dir.name] = i

print(f'Classes : {len(class_names)}')
print(f'Sample  : {class_names[:8]}')

# ── Convert bb_info.txt → YOLO labels ──────────────────────────────────────
# bb_info.txt format (space-separated, 1 header row):
#   img    x1    y1    x2    y2
#   1      52    36    253   317
#   3      16     1    500   374
#
# Coordinates are pixel values; (x1,y1) = top-left, (x2,y2) = bottom-right
# (inclusive, 1-indexed).  Multiple rows for the same img_id mean multiple
# instances of that food in the same image.

random.seed(42)
VAL_SPLIT = 0.15
skipped_missing = 0
skipped_corrupt = 0
skipped_degenerate = 0

for cat_dir in sorted_cat_dirs:
    cls_idx = cat_id_map.get(cat_dir.name)
    if cls_idx is None:
        continue

    bb_file = cat_dir / 'bb_info.txt'
    if not bb_file.exists():
        continue

    lines = bb_file.read_text(encoding='utf-8', errors='replace').splitlines()

    # Collect all boxes per image (skip header row)
    boxes_by_img: dict[str, list[tuple[int, int, int, int]]] = {}
    for line in lines[1:]:
        parts = line.split()          # handles any mix of spaces/tabs
        if len(parts) < 5:
            continue
        img_id = parts[0]
        try:
            x1, y1, x2, y2 = int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
        except ValueError:
            continue
        # Guard: skip degenerate boxes before any further processing
        if x2 <= x1 or y2 <= y1:
            skipped_degenerate += 1
            continue
        boxes_by_img.setdefault(img_id, []).append((x1, y1, x2, y2))

    for img_id, boxes in boxes_by_img.items():
        img_path = cat_dir / f'{img_id}.jpg'
        if not img_path.exists():
            skipped_missing += 1
            continue

        try:
            with Image.open(img_path) as im:
                W, H = im.size
        except Exception:
            skipped_corrupt += 1
            continue

        if W == 0 or H == 0:
            skipped_corrupt += 1
            continue

        split = 'val' if random.random() < VAL_SPLIT else 'train'
        # Prefix with folder number to avoid cross-category filename collisions
        unique_stem = f'{cat_dir.name}_{img_id}'
        dst_img = YOLO_ROOT / 'images' / split / f'{unique_stem}.jpg'
        dst_lbl = YOLO_ROOT / 'labels' / split / f'{unique_stem}.txt'

        shutil.copy2(img_path, dst_img)

        yolo_lines = []
        for (x1, y1, x2, y2) in boxes:
            # Convert pixel coords → normalised YOLO centre + dimensions
            cx = ((x1 + x2) / 2.0) / W
            cy = ((y1 + y2) / 2.0) / H
            w  = (x2 - x1) / W
            h  = (y2 - y1) / H
            # Clamp to [0, 1] in case coordinates slightly exceed image bounds
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))
            if w <= 0 or h <= 0:
                skipped_degenerate += 1
                continue
            yolo_lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        if yolo_lines:
            dst_lbl.write_text('\n'.join(yolo_lines) + '\n')
        else:
            # No valid boxes — remove the copied image
            dst_img.unlink(missing_ok=True)

n_train = len(list((YOLO_ROOT / 'images' / 'train').glob('*.jpg')))
n_val   = len(list((YOLO_ROOT / 'images' / 'val').glob('*.jpg')))
print(f'\nConversion complete.')
print(f'  Train images    : {n_train}')
print(f'  Val images      : {n_val}')
print(f'  Skipped missing : {skipped_missing}')
print(f'  Skipped corrupt : {skipped_corrupt}')
print(f'  Degenerate boxes: {skipped_degenerate}')

Classes : 256
Sample  : ['rice', 'eels_on_rice', 'pilaf', "chicken-'n'-egg_on_rice", 'pork_cutlet_on_rice', 'beef_curry', 'sushi', 'chicken_rice']

Conversion complete.
  Train images    : 26661
  Val images      : 4734
  Skipped missing : 0
  Skipped corrupt : 0
  Degenerate boxes: 0


## 5 — Write data.yaml

In [ ]:
import yaml

data_yaml = {
    'path': str(YOLO_ROOT),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(class_names),
    'names': class_names,
}

yaml_path = YOLO_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

print(f'data.yaml written to {yaml_path}')
print(f'  nc    = {data_yaml["nc"]}')
print(f'  names[:5] = {class_names[:5]}')

data.yaml written to /content/uec_yolo/data.yaml
  nc    = 256
  names[:5] = ['rice', 'eels_on_rice', 'pilaf', "chicken-'n'-egg_on_rice", 'pork_cutlet_on_rice']


## 6 — Train YOLOv8n for 50 Epochs

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # downloads pretrained weights automatically

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name='nutriscan_v1',
    project='/content/runs',
    exist_ok=True,
    patience=10,          # early stopping
    cache=True,
    device=0,             # GPU
)

print('Training complete.')
print(f'Best weights: {results.save_dir}/weights/best.pt')

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/uec_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=nutriscan_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10,

## 7 — Save best.pt to Google Drive

In [ ]:
import shutil
from pathlib import Path

best_src = Path(results.save_dir) / 'weights' / 'best.pt'
best_dst = Path(DRIVE_DIR) / 'best.pt'

shutil.copy2(best_src, best_dst)
print(f'Saved best.pt → {best_dst}')

# Also copy training metrics
metrics_src = Path(results.save_dir) / 'results.csv'
if metrics_src.exists():
    shutil.copy2(metrics_src, Path(DRIVE_DIR) / 'results.csv')
    print('Saved results.csv')

Saved best.pt → /content/drive/MyDrive/nutriscan/best.pt
Saved results.csv


## 8 — Quick Validation

In [ ]:
best_model = YOLO(str(best_dst))
metrics = best_model.val(data=str(yaml_path), imgsz=640, device=0)

print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')

Mounted at /content/drive


NameError: name 'yaml_path' is not defined